In [ ]:
# 13 — Gold: handyman scorecards
# One row per (handyman, incident type) plus an overall row (incident_type = 'all'): completed / cancelled /
# active jobs, success rate (share of rated jobs rated 4–5), average rating, average resolution time and
# travel time, billed hours and amounts, and the hourly rate (amount / hours over the latest 50 billed jobs).
# Full recompute from silver; the live stream updates single handymen incrementally.

In [ ]:
import os, sys, time
sys.path.insert(0, os.getcwd())          # pipeline_lib.py lives next to this notebook
import pipeline_lib as pl
from pyspark.sql import functions as F

dbutils.widgets.text("run_id", "manual")
RUN_ID = dbutils.widgets.get("run_id")          # tables are created by 10_export_raw

started = time.time()
pl.compute_performance(spark.table(pl.T_SILVER)).createOrReplaceTempView("_perf")
spark.sql(f"""CREATE OR REPLACE TABLE {pl.T_GOLD_PERFORMANCE} CLUSTER BY (handyman_user_id)
              COMMENT 'Handyman scorecards per incident type (+ overall row incident_type = all)'
              AS SELECT * FROM _perf""")

gold = spark.table(pl.T_GOLD_PERFORMANCE)
expected = spark.table(pl.T_SILVER).agg(F.count_distinct("handyman_user_id")).first()[0]
# One pass over the scorecards for all checks. Billed jobs come from completed jobs only, and a rate exists
# exactly when there are billed hours
n_rows, n_handymen, bad_rates, bad_billing, rated = gold.agg(
    F.count("*"),
    F.count_if(F.col("incident_type") == "all"),
    F.count_if(F.expr("success_rate < 0 OR success_rate > 1 OR avg_rating < 1 OR avg_rating > 5")),
    F.count_if(F.expr("billed_jobs > jobs_completed OR (hourly_rate IS NULL) != (billed_jobs = 0) "
                      "OR hourly_rate < 10 OR hourly_rate > 300")),
    F.count_if(F.expr("incident_type = 'all' AND hourly_rate IS NOT NULL"))).first()
display(gold.where("incident_type = 'all'").orderBy(F.desc("jobs_completed")).limit(10))

pl.log_step(spark, RUN_ID, "13_gold_performance", started, rows_in=None, rows_out=n_rows,
            checks={"every_handyman_has_overall_row": n_handymen == expected,
                    "rates_in_range": bad_rates == 0,
                    "billing_consistent": bad_billing == 0,
                    "most_handymen_have_hourly_rate": rated >= 0.9 * n_handymen})